In [1]:
# ONE-CELL NON-VISUAL RUNNER
# Generated from eredivisie_shot_analytics.ipynb. This trains/models/saves data only; no plots or PNG visuals are created.
RUN_PYMC_HIERARCHICAL = False  # Optional original notebook block; slow and not needed for saved CSV/model outputs.

try:
    from IPython.display import display
except Exception:
    def display(obj):
        try:
            print(obj.to_string())
        except Exception:
            print(obj)


# === Original notebook cell 2 ===
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

for pkg, imp in [('xgboost','xgboost'), ('scikit-learn','sklearn'), ('joblib','joblib'), ('scipy','scipy')]:
    try:
        __import__(imp)
    except ImportError:
        _pip(pkg)

print('Dependencies ready ✓')

# === Original notebook cell 3 ===
import os, glob, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_curve, brier_score_loss, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy import stats
from scipy.special import betaln
from scipy.optimize import minimize
import joblib
import xgboost as xgb

print('All non-visual packages loaded ✓')

# === Original notebook cell 5 ===
# ── CONFIGURE THESE PATHS ────────────────────────────────────────────────────
SEASON_ROOTS = [
    '/Users/marclamberts/Event data/CZ/DONE',
    '/Users/marclamberts/Event data/Eredivisie/',
    '/Users/marclamberts/Event data/Belgium/',
    '/Users/marclamberts/Event data/Croatia/',
    '/Users/marclamberts/Event data/Bundelsiga/',
    '/Users/marclamberts/Event data/Ligue 1/',
    '/Users/marclamberts/Event data/La Liga/',
    '/Users/marclamberts/Event data/Premier League/',
    '/Users/marclamberts/Event data/Serie A/',

]
OUTPUT_DIR = '/Users/marclamberts/Downloads/Danger Model/xg_output'

# Big Chance (Q214) is a documented but subjective Opta analyst judgement.
# Keep False for the portable/objective xG model; set True for an Opta-enriched variant.
USE_BIG_CHANCE = False
PENALTY_XG = 0.79
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Visual palette
FIG_BG   = '#0e1117'
PITCH_BG = '#0d1117'
LINE_COL = '#c9d1d9'
C_BLUE   = '#58a6ff'
C_ORANGE = '#f78166'
C_GREEN  = '#3fb950'
C_YELLOW = '#e3b341'
C_PURPLE = '#bc8cff'
C_MUTED  = '#484f58'

# Pitch / goal constants (Opta coordinates)
GOAL_X        = 100.0
GOAL_Y_LEFT   = 44.0
GOAL_Y_RIGHT  = 56.0
GOAL_Y_CENTRE = 50.0
GOAL_WIDTH    = 12.0
GOAL_H_HIGH   = 62.0

# Opta qualifiers from:
# https://github.com/tomh05/football-scores/blob/master/data/reference/opta-qualifiers.csv
# Only documented meanings are used. Deprecated fields are retained only where
# they genuinely occur in historical data.
Q = dict(
    PENALTY=9,
    HEADER=15,
    RIGHT_FOOT=20,
    OTHER_BODY_PART=21,
    REGULAR_PLAY=22,
    FAST_BREAK=23,
    SET_PIECE=24,
    FROM_CORNER=25,
    DIRECT_FREE_KICK=26,
    ASSISTED=29,
    RELATED_EVENT_ID=55,
    LEFT_FOOT=72,
    BLOCKED=82,
    ONE_ON_ONE=89,
    CORNER_SECOND_PHASE=96,
    GOAL_Y=102,
    GOAL_HEIGHT=103,
    VOLLEY=108,
    OVERHEAD=109,
    HALF_VOLLEY=110,
    SCRAMBLE=112,
    LOB=117,
    DEFLECTION=133,
    KEEPER_SAVED_OFF_TARGET=137,
    INTENTIONAL_ASSIST=154,
    THROW_IN_SET_PIECE=160,
    PULL_BACK=195,
    BIG_CHANCE=214,
    INDIVIDUAL_PLAY=215,
)

print(f'Output dir: {OUTPUT_DIR}')
print(f'xG variant: {"Opta-enriched" if USE_BIG_CHANCE else "objective (Big Chance excluded)"}')

# === Original notebook cell 7 ===
def open_angle(x, y):
    shot  = np.array([x, y], dtype=float)
    left  = np.array([GOAL_X, GOAL_Y_LEFT],  dtype=float)
    right = np.array([GOAL_X, GOAL_Y_RIGHT], dtype=float)
    v1, v2 = left - shot, right - shot
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0: return 0.0
    return float(np.degrees(np.arccos(np.clip(np.dot(v1,v2)/(n1*n2),-1,1))))

def dist_to_goal(x, y):
    return float(np.sqrt((GOAL_X-x)**2 + (y-GOAL_Y_CENTRE)**2))

def placement_score(gy_norm, gh_norm):
    if pd.isna(gy_norm) or pd.isna(gh_norm): return np.nan
    gy = float(np.clip(gy_norm, -1, 1))
    gh = float(np.clip(gh_norm,  0, 1))
    keeper_h = 0.40
    lat  = abs(gy)
    vert = abs(gh - keeper_h) / max(keeper_h, 1 - keeper_h)
    return float(np.clip(np.sqrt(0.6*lat**2 + 0.4*vert**2)*100, 0, 100))

def goal_zone(goal_y, goal_h):
    if pd.isna(goal_y) or pd.isna(goal_h): return 'Unknown'
    h = 'Top' if goal_h >= GOAL_H_HIGH else 'Bottom'
    s = 'Left' if goal_y < 47.5 else ('Right' if goal_y > 52.5 else 'Centre')
    return f'{h} {s}'

def get_q(event, qid):
    for q in event.get('qualifier', []):
        if q['qualifierId'] == qid: return q.get('value', 1)
    return None

def has_q(event, qid):
    return any(q['qualifierId'] == qid for q in event.get('qualifier', []))

def safe_float(v):
    try: return float(v)
    except: return np.nan

def season_from_path(path):
    for p in path.replace('\\','/').split('/'):
        if 'Eredivisie' in p or 'eredivisie' in p: return p
        if p.startswith('WSL'): return p
    return 'Unknown'

print('Helpers defined ✓')

# === Original notebook cell 8 ===
def _event_sort_key(event, fallback_index=0):
    return (
        int(event.get('periodId') or 0),
        int(event.get('timeMin') or 0),
        int(event.get('timeSec') or 0),
        int(event.get('eventId') or fallback_index),
    )


def _finish_outcome(type_id, keeper_save=False, saved_off_target=False, outfield_block=False):
    if str(type_id) == '16':
        return 'goal'
    if str(type_id) == '14':
        return 'post'
    if keeper_save:
        return 'saved'
    if saved_off_target:
        return 'saved_off_target'
    if outfield_block:
        return 'blocked'
    return 'off_target'


def _body_part(row):
    if row.get('is_header', 0):
        return 'header'
    if row.get('is_right_foot', 0):
        return 'right_foot'
    if row.get('is_left_foot', 0):
        return 'left_foot'
    if row.get('is_other_body_part', 0):
        return 'other_body_part'
    return 'unknown'


def _related_event_lookup(events):
    exact = {}
    loose = {}
    global_id = {}
    for idx, event in enumerate(events):
        event_id = str(event.get('eventId', ''))
        period_id = int(event.get('periodId') or 0)
        contestant_id = str(event.get('contestantId', ''))
        if event_id:
            exact[(period_id, contestant_id, event_id)] = event
            loose.setdefault((period_id, event_id), []).append(event)
        gid = str(event.get('id', ''))
        if gid:
            global_id[gid] = event
    return exact, loose, global_id


def _find_related_event(event, related_event_id, exact_lookup, loose_lookup, global_lookup):
    related_event_id = str(related_event_id or '').strip()
    if not related_event_id:
        return None
    period_id = int(event.get('periodId') or 0)
    contestant_id = str(event.get('contestantId', ''))
    related = exact_lookup.get((period_id, contestant_id, related_event_id))
    if related is not None:
        return related
    candidates = loose_lookup.get((period_id, related_event_id), [])
    same_team = [candidate for candidate in candidates if str(candidate.get('contestantId', '')) == contestant_id]
    if same_team:
        return same_team[0]
    if candidates:
        return candidates[0]
    return global_lookup.get(related_event_id)


def _score_state_before_events(events):
    contestants = []
    for event in events:
        contestant_id = str(event.get('contestantId', ''))
        if contestant_id and contestant_id not in contestants:
            contestants.append(contestant_id)

    goals_by_team = {team: 0 for team in contestants}
    states = {}
    for idx, event in sorted(enumerate(events), key=lambda item: _event_sort_key(item[1], item[0])):
        event_key = str(event.get('id', event.get('eventId', idx)))
        team_id = str(event.get('contestantId', ''))
        opponent_ids = [team for team in contestants if team != team_id]
        opponent_id = opponent_ids[0] if opponent_ids else ''
        team_goals = int(goals_by_team.get(team_id, 0))
        opp_goals = int(goals_by_team.get(opponent_id, 0)) if opponent_id else 0
        diff = team_goals - opp_goals
        states[event_key] = {
            'opponent_contestant_id': opponent_id,
            'team_goals_before': team_goals,
            'opponent_goals_before': opp_goals,
            'goal_diff_before': diff,
            'game_state': 'winning' if diff > 0 else ('losing' if diff < 0 else 'drawing'),
        }
        if str(event.get('typeId')) == '16':
            goals_by_team[team_id] = goals_by_team.get(team_id, 0) + 1
    return states


def load_match(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        data = json.load(f)
    events = data.get('liveData', {}).get('event', data.get('event', []))
    rows = []
    match_id = os.path.normpath(os.path.abspath(path))
    match_file = os.path.basename(path)
    match_date = pd.to_datetime(match_file[:10], errors='coerce')
    match_name = os.path.splitext(match_file)[0]
    home_team_name, away_team_name = (match_name[11:].split(' - ', 1) + [''])[:2] if len(match_name) > 11 else ('', '')

    exact_related, loose_related, global_related = _related_event_lookup(events)
    score_states = _score_state_before_events(events)

    for e in events:
        tid = str(e.get('typeId'))
        if tid not in ('13', '14', '15', '16'):
            continue
        x = safe_float(e.get('x'))
        y = safe_float(e.get('y'))
        if np.isnan(x) or np.isnan(y):
            continue

        gy_raw  = safe_float(get_q(e, Q['GOAL_Y']))
        gh_raw  = safe_float(get_q(e, Q['GOAL_HEIGHT']))
        gy_norm = (gy_raw - GOAL_Y_CENTRE) / (GOAL_WIDTH / 2) if not np.isnan(gy_raw) else np.nan
        gh_norm = gh_raw / 100.0 if not np.isnan(gh_raw) else np.nan

        if not np.isnan(gy_norm) and not np.isnan(gh_norm):
            gf_dist = np.sqrt(gy_norm**2 + (gh_norm - 0.35)**2)
            corner = int(abs(gy_norm) > 0.55 or gh_norm > 0.60)
            ps = placement_score(gy_norm, gh_norm)
        elif not np.isnan(gy_norm):
            gf_dist = abs(gy_norm)
            corner = int(abs(gy_norm) > 0.55)
            ps = abs(gy_norm) * 100
        else:
            gf_dist = corner = ps = np.nan

        dist = dist_to_goal(x, y)
        angle = open_angle(x, y)
        y_sym = abs(y - GOAL_Y_CENTRE)

        outfield_block = tid == '15' and has_q(e, Q['BLOCKED'])
        saved_off_target = tid == '15' and has_q(e, Q['KEEPER_SAVED_OFF_TARGET'])
        keeper_save = tid == '15' and not outfield_block and not saved_off_target
        on_target = tid == '16' or keeper_save

        related_event_id = str(get_q(e, Q['RELATED_EVENT_ID']) or '').strip()
        related = _find_related_event(e, related_event_id, exact_related, loose_related, global_related)
        related_end_x = safe_float(get_q(related, 140)) if related else np.nan
        related_end_y = safe_float(get_q(related, 141)) if related else np.nan
        has_shot_assist = bool(has_q(e, Q['ASSISTED']) or related_event_id or related)

        event_key = str(e.get('id', e.get('eventId', '')))
        state = score_states.get(event_key, {})
        team_goals_after = state.get('team_goals_before', 0) + int(tid == '16')
        opponent_goals_after = state.get('opponent_goals_before', 0)

        row = {
            'match_id': match_id,
            'match_file': match_file,
            'match_date': match_date,
            'match_name': match_name,
            'home_team_name': home_team_name,
            'away_team_name': away_team_name,
            'event_id': str(e.get('id', e.get('eventId', ''))),
            'event_index': int(e.get('eventId') or 0),
            'season': season_from_path(path),
            'player_id': str(e.get('playerId', '')),
            'player_name': e.get('playerName', ''),
            'contestant_id': str(e.get('contestantId', '')),
            'opponent_contestant_id': state.get('opponent_contestant_id', ''),
            'type_id': int(tid),
            'period_id': int(e.get('periodId') or 0),
            'time_min': int(e.get('timeMin') or 0),
            'time_sec': int(e.get('timeSec') or 0),
            'time_seconds': int(e.get('timeMin') or 0) * 60 + int(e.get('timeSec') or 0),
            'timestamp': e.get('timeStamp', ''),
            'team_goals_before': state.get('team_goals_before', 0),
            'opponent_goals_before': state.get('opponent_goals_before', 0),
            'goal_diff_before': state.get('goal_diff_before', 0),
            'game_state': state.get('game_state', 'drawing'),
            'team_goals_after': team_goals_after,
            'opponent_goals_after': opponent_goals_after,
            'goal_diff_after': team_goals_after - opponent_goals_after,
            'x': x, 'y': y, 'y_sym': y_sym,
            'distance': dist, 'log_distance': np.log(max(dist, 0.5)),
            'angle': angle, 'angle_sin': np.sin(np.radians(angle)),
            'in_six_yard': int(x >= 94.2 and 36.8 <= y <= 63.2),
            'in_penalty_box': int(x >= 83.0 and 21.1 <= y <= 78.9),
            'central_y': int(y_sym < GOAL_WIDTH / 2),
            'dist_to_post': abs(y_sym - GOAL_WIDTH / 2),
            'goal_y_raw': gy_raw, 'goal_y_norm': gy_norm,
            'goal_h_raw': gh_raw, 'goal_h_norm': gh_norm,
            'goal_frame_dist': gf_dist, 'corner_zone': corner,
            'placement_score': ps, 'goal_zone': goal_zone(gy_raw, gh_raw),
            'related_event_id': related_event_id,
            'related_event_global_id': str(related.get('id', '')) if related else '',
            'related_type_id': int(related.get('typeId') or 0) if related else 0,
            'related_player_id': str(related.get('playerId', '')) if related else '',
            'related_player_name': related.get('playerName', '') if related else '',
            'related_contestant_id': str(related.get('contestantId', '')) if related else '',
            'related_x': safe_float(related.get('x')) if related else np.nan,
            'related_y': safe_float(related.get('y')) if related else np.nan,
            'related_end_x': related_end_x,
            'related_end_y': related_end_y,
            'related_time_min': int(related.get('timeMin') or 0) if related else 0,
            'related_time_sec': int(related.get('timeSec') or 0) if related else 0,
            'related_outcome': int(related.get('outcome') or 0) if related else 0,
            'shot_assist_player_id': str(related.get('playerId', '')) if related else '',
            'shot_assist_player_name': related.get('playerName', '') if related else '',
            'shot_assist_type_id': int(related.get('typeId') or 0) if related else 0,
            'has_shot_assist': int(has_shot_assist),
            'is_header': int(has_q(e, Q['HEADER'])),
            'is_right_foot': int(has_q(e, Q['RIGHT_FOOT'])),
            'is_left_foot': int(has_q(e, Q['LEFT_FOOT'])),
            'is_other_body_part': int(has_q(e, Q['OTHER_BODY_PART'])),
            'is_volley': int(has_q(e, Q['VOLLEY'])),
            'is_half_volley': int(has_q(e, Q['HALF_VOLLEY'])),
            'is_overhead': int(has_q(e, Q['OVERHEAD'])),
            'is_lob': int(has_q(e, Q['LOB'])),
            'is_deflected': int(has_q(e, Q['DEFLECTION'])),
            'is_big_chance': int(has_q(e, Q['BIG_CHANCE'])),
            'is_one_on_one': int(has_q(e, Q['ONE_ON_ONE'])),
            'is_fast_break': int(has_q(e, Q['FAST_BREAK'])),
            'is_from_corner': int(has_q(e, Q['FROM_CORNER'])),
            'is_corner_second_phase': int(has_q(e, Q['CORNER_SECOND_PHASE'])),
            'is_free_kick': int(has_q(e, Q['DIRECT_FREE_KICK'])),
            'is_penalty': int(has_q(e, Q['PENALTY'])),
            'is_set_piece': int(has_q(e, Q['SET_PIECE'])),
            'is_throw_in_set_piece': int(has_q(e, Q['THROW_IN_SET_PIECE'])),
            'is_open_play': int(has_q(e, Q['REGULAR_PLAY'])),
            # Pull Back and Intentional Assist describe the creating pass. Check
            # both the shot and the related creating event when the feed links it.
            'is_pull_back': int(has_q(e, Q['PULL_BACK']) or (related is not None and has_q(related, Q['PULL_BACK']))),
            'is_assisted': int(has_q(e, Q['ASSISTED'])),
            'is_intentional_assist': int(has_q(e, Q['INTENTIONAL_ASSIST']) or (related is not None and has_q(related, Q['INTENTIONAL_ASSIST']))),
            'is_individual_play': int(has_q(e, Q['INDIVIDUAL_PLAY'])),
            'is_scramble': int(has_q(e, Q['SCRAMBLE'])),
            'is_goal': int(tid == '16'),
            'is_on_target': int(on_target),
            'is_post': int(tid == '14'),
            'is_blocked': int(outfield_block),
            'is_outfield_block': int(outfield_block),
            'is_keeper_save': int(keeper_save),
            'is_saved_off_target': int(saved_off_target),
            'qualifier_ids': ','.join(str(q.get('qualifierId')) for q in e.get('qualifier', [])),
        }
        row['shot_body_part'] = _body_part(row)
        row['finish_outcome'] = _finish_outcome(tid, keeper_save, saved_off_target, outfield_block)
        row['shot_suppression_team_id'] = row['opponent_contestant_id']
        row['shot_suppression_value'] = 1
        row['xg_suppression_value'] = np.nan
        rows.append(row)
    return pd.DataFrame(rows)

print('load_match defined with game state, shot-assist links and suppression context ✓')

# === Original notebook cell 10 ===
json_files = []
for root in SEASON_ROOTS:
    json_files.extend(glob.glob(os.path.join(root, '**', '*.json'), recursive=True))
    json_files.extend(glob.glob(os.path.join(root, '*.json')))
json_files = sorted(set(os.path.abspath(p) for p in json_files))
print(f'Found {len(json_files)} JSON files across {len(SEASON_ROOTS)} season roots')

frames = []
errors = []
for path in json_files:
    try:
        frame = load_match(path)
        if not frame.empty:
            frames.append(frame)
    except Exception as exc:
        errors.append((os.path.basename(path), str(exc)))

if errors:
    print(f'Skipped {len(errors)} files with errors')
if not frames:
    raise RuntimeError('No valid shot data were loaded. Check SEASON_ROOTS.')

shots = pd.concat(frames, ignore_index=True)

# Remove exact duplicate event records when the provider supplies a stable ID.
has_event_id = shots['event_id'].astype(str).ne('')
shots_with_id = shots[has_event_id].drop_duplicates(['match_id', 'event_id'])
shots_without_id = shots[~has_event_id].drop_duplicates(
    ['match_id', 'period_id', 'time_min', 'player_id', 'x', 'y', 'type_id']
)
shots = pd.concat([shots_with_id, shots_without_id], ignore_index=True)

# Descriptive technique index only; it is not an xG model input.
def technique_index(row):
    s = 0
    if row['is_header']:       s += 20
    if row['is_volley']:       s += 15
    if row['is_half_volley']:  s += 10
    if row['is_overhead']:     s += 25
    if row['is_lob']:          s += 10
    return float(min(s, 100))

shots['technique_index'] = shots.apply(technique_index, axis=1)

print(f"Loaded {len(shots):,} shots | Goals: {shots['is_goal'].sum():,} "
      f"({shots['is_goal'].mean()*100:.1f}%)")
print(f"Unique matches: {shots['match_id'].nunique():,}")
print(f"Seasons: {sorted(shots['season'].unique())}")
print(shots.head(3).to_string(index=False))

# === Original notebook cell 12 ===
class CalibratedXGB:
    """XGBoost plus held-out isotonic probability calibration."""
    def __init__(self, clf, iso):
        self.clf = clf
        self.iso = iso

    def predict_proba(self, X):
        raw = self.clf.predict_proba(X)[:, 1]
        calibrated = np.clip(self.iso.predict(raw), 0.001, 0.999)
        return np.column_stack([1 - calibrated, calibrated])


def expected_calibration_error(y_true, probabilities, bins=10):
    table = pd.DataFrame({'y': np.asarray(y_true), 'p': probabilities})
    table['bin'] = pd.cut(table['p'], np.linspace(0, 1, bins + 1),
                          include_lowest=True, duplicates='drop')
    grouped = table.groupby('bin', observed=False).agg(
        n=('y', 'size'), observed=('y', 'mean'), predicted=('p', 'mean')
    ).dropna()
    return float(
        ((grouped['n'] / grouped['n'].sum())
         * (grouped['observed'] - grouped['predicted']).abs()).sum()
    )


def evaluate_probabilities(y_true, probabilities):
    return {
        'auc': roc_auc_score(y_true, probabilities),
        'log_loss': log_loss(y_true, probabilities, labels=[0, 1]),
        'brier': brier_score_loss(y_true, probabilities),
        'ece_10': expected_calibration_error(y_true, probabilities, bins=10),
        'predicted_goals': float(np.sum(probabilities)),
        'actual_goals': int(np.sum(y_true)),
    }


def train_model(X_tr, y_tr, g_tr, X_cal, y_cal, X_test, y_test, label):
    group_cv = GroupKFold(n_splits=5)
    grid = GridSearchCV(
        xgb.XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=42,
            verbosity=0,
            n_jobs=1,
        ),
        {
            'max_depth': [3, 4, 5],
            'learning_rate': [0.03, 0.05],
            'n_estimators': [300, 600],
            'subsample': [0.8],
            'colsample_bytree': [0.8],
            'min_child_weight': [10, 25],
            # Probability estimation: do not rebalance the goal class.
            'scale_pos_weight': [1.0],
        },
        scoring='neg_log_loss',
        cv=group_cv,
        n_jobs=-1,
        verbose=0,
    )
    grid.fit(X_tr, y_tr, groups=g_tr)
    clf = grid.best_estimator_

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(clf.predict_proba(X_cal)[:, 1], y_cal)
    model = CalibratedXGB(clf, iso)

    p_test = model.predict_proba(X_test)[:, 1]
    metrics = evaluate_probabilities(y_test, p_test)
    fpr, tpr, _ = roc_curve(y_test, p_test)
    print(
        f"  [{label}] CV log-loss: {-grid.best_score_:.4f} | "
        f"Test log-loss: {metrics['log_loss']:.4f} | "
        f"Brier: {metrics['brier']:.4f} | "
        f"AUC: {metrics['auc']:.4f} | ECE: {metrics['ece_10']:.4f}"
    )
    print(f"  Actual goals: {metrics['actual_goals']:,} | "
          f"Predicted goals: {metrics['predicted_goals']:.1f}")
    return model, p_test, fpr, tpr, metrics['auc'], metrics['brier'], metrics


# Penalties are kept outside the non-penalty model and assigned a documented
# fixed baseline. This prevents the held-out test set influencing penalty xG.
pen_mask = shots['is_penalty'].eq(1)
nonpen = shots[~pen_mask].copy().reset_index(drop=True)
pen_xg = PENALTY_XG
for col in ['xg', 'psxg', 'situation_danger', 'p_ontarget', 'xgot']:
    shots.loc[pen_mask, col] = pen_xg
print(f'Penalty xG (fixed): {pen_xg:.3f}  n={pen_mask.sum()}')

# Geometry features. XGBoost can learn non-linear effects, so the final model
# avoids large collections of duplicate polynomial transformations.
nonpen['x_angle'] = nonpen['x'] * nonpen['angle_sin']
nonpen['dist_sq'] = nonpen['distance'] ** 2
nonpen['zone_angle'] = nonpen['in_penalty_box'] * nonpen['angle']
for col in ['x_angle', 'dist_sq', 'zone_angle']:
    shots[col] = np.nan
shots.loc[~pen_mask, ['x_angle', 'dist_sq', 'zone_angle']] = (
    nonpen[['x_angle', 'dist_sq', 'zone_angle']].values
)

XG_CORE_FEATURES = [
    'distance', 'angle', 'y_sym',
    'is_header', 'is_right_foot', 'is_left_foot', 'is_other_body_part',
    'is_volley', 'is_half_volley', 'is_overhead', 'is_lob',
    'is_one_on_one', 'is_fast_break', 'is_from_corner',
    'is_corner_second_phase', 'is_free_kick', 'is_set_piece',
    'is_throw_in_set_piece', 'is_open_play', 'is_assisted',
    'is_intentional_assist', 'is_individual_play', 'is_scramble',
]
XG_FEATURES = XG_CORE_FEATURES + (['is_big_chance'] if USE_BIG_CHANCE else [])

# Post-shot features remain separate. Deflection and goalmouth coordinates
# must never enter the pre-shot xG model.
PSXG_FEATURES = XG_FEATURES + [
    'is_deflected', 'goal_y_norm', 'goal_h_norm',
    'goal_frame_dist', 'corner_zone', 'placement_score',
]
SIT_FEATURES = [
    'is_big_chance', 'is_one_on_one', 'is_fast_break',
    'is_from_corner', 'is_corner_second_phase', 'is_free_kick',
    'is_set_piece', 'is_throw_in_set_piece', 'is_open_play',
    'is_assisted', 'is_intentional_assist', 'is_individual_play',
    'is_scramble', 'is_header', 'period_id', 'time_min',
]
POT_FEATURES = XG_FEATURES.copy()

# Fail loudly if a mapped qualifier is outcome-coded or absent. Zero-count
# fields are dropped rather than silently becoming useless model columns.
feature_counts = nonpen[XG_FEATURES].sum().sort_values()
zero_features = feature_counts[feature_counts.eq(0)].index.tolist()
if zero_features:
    print('Dropping zero-count xG features:', zero_features)
    XG_FEATURES = [c for c in XG_FEATURES if c not in zero_features]
    PSXG_FEATURES = [c for c in PSXG_FEATURES if c not in zero_features]
    POT_FEATURES = [c for c in POT_FEATURES if c not in zero_features]

audit_rows = []
for col in XG_FEATURES:
    flagged = nonpen[col].eq(1) if set(nonpen[col].dropna().unique()).issubset({0, 1}) else pd.Series(False, index=nonpen.index)
    audit_rows.append({
        'feature': col,
        'nonzero_n': int(nonpen[col].ne(0).sum()),
        'flagged_goal_rate': float(nonpen.loc[flagged, 'is_goal'].mean()) if flagged.any() else np.nan,
    })
feature_audit = pd.DataFrame(audit_rows)
print(feature_audit.to_string(index=False))

binary_audit = feature_audit.dropna(subset=['flagged_goal_rate'])
bad = binary_audit[
    (binary_audit['nonzero_n'] >= 25)
    & ((binary_audit['flagged_goal_rate'] <= 0.001)
       | (binary_audit['flagged_goal_rate'] >= 0.999))
]
if not bad.empty:
    raise ValueError(
        'Probable qualifier leakage detected. Inspect these features:\n'
        + bad.to_string(index=False)
    )

X_base = nonpen[XG_FEATURES].fillna(0).astype(float)
X_sit = nonpen[SIT_FEATURES].fillna(0).astype(float)
X_pot = nonpen[POT_FEATURES].fillna(0).astype(float)
y_goal = nonpen['is_goal'].astype(int)
y_ot = nonpen['is_on_target'].astype(int)
groups = nonpen['match_id']

# Prefer a chronological 60/20/20 match split. This tests whether the model
# transfers to future matches instead of merely reproducing a random sample.
match_dates = nonpen[['match_id', 'match_date']].drop_duplicates('match_id')
dated_share = match_dates['match_date'].notna().mean()
if dated_share >= 0.80 and len(match_dates) >= 20:
    ordered_matches = (
        match_dates.dropna().sort_values(['match_date', 'match_id'])['match_id'].tolist()
    )
    undated_matches = match_dates.loc[match_dates['match_date'].isna(), 'match_id'].tolist()
    ordered_matches = undated_matches + ordered_matches
    train_end = int(len(ordered_matches) * 0.60)
    cal_end = int(len(ordered_matches) * 0.80)
    train_matches = set(ordered_matches[:train_end])
    cal_matches = set(ordered_matches[train_end:cal_end])
    test_matches = set(ordered_matches[cal_end:])
    train_idx = np.flatnonzero(groups.isin(train_matches))
    cal_idx = np.flatnonzero(groups.isin(cal_matches))
    test_idx = np.flatnonzero(groups.isin(test_matches))
    split_method = 'chronological 60/20/20 by match date'
else:
    gss_outer = GroupShuffleSplit(1, test_size=0.20, random_state=42)
    dev_idx, test_idx = next(gss_outer.split(X_base, y_goal, groups=groups))
    gss_inner = GroupShuffleSplit(1, test_size=0.25, random_state=0)
    tr_rel, cal_rel = next(gss_inner.split(
        X_base.iloc[dev_idx], y_goal.iloc[dev_idx], groups=groups.iloc[dev_idx]
    ))
    train_idx = dev_idx[tr_rel]
    cal_idx = dev_idx[cal_rel]
    split_method = 'fallback match-grouped random 60/20/20'

def split(X):
    return X.iloc[train_idx], X.iloc[cal_idx], X.iloc[test_idx]

X_base_tr, X_base_cal, X_base_test = split(X_base)
X_sit_tr, X_sit_cal, X_sit_test = split(X_sit)
X_pot_tr, X_pot_cal, X_pot_test = split(X_pot)
y_tr, y_cal, y_test = y_goal.iloc[train_idx], y_goal.iloc[cal_idx], y_goal.iloc[test_idx]
y_ot_tr, y_ot_cal, y_ot_test = y_ot.iloc[train_idx], y_ot.iloc[cal_idx], y_ot.iloc[test_idx]
g_tr = groups.iloc[train_idx]

print(f'Split: {split_method}')
print(f'Train: {len(y_tr):,}  Cal: {len(y_cal):,}  Test: {len(y_test):,}')
print(f'Features — xG: {len(XG_FEATURES)}  psxG: {len(PSXG_FEATURES)}  P(OT): {len(POT_FEATURES)}')

# === Original notebook cell 14 ===
print('── Model 1: Pre-Shot xG ──')
xg_model, xg_p, xg_fpr, xg_tpr, xg_auc, xg_brier, xg_metrics = train_model(
    X_base_tr, y_tr, g_tr, X_base_cal, y_cal, X_base_test, y_test, 'xG')
nonpen['xg'] = xg_model.predict_proba(X_base)[:, 1]
shots.loc[~pen_mask, 'xg'] = nonpen['xg'].values
joblib.dump(xg_model, os.path.join(OUTPUT_DIR, 'model_xg.pkl'))

print('\n── Model 2: Post-Shot psxG ──')
has_pl  = nonpen[['goal_y_norm','goal_h_norm']].notna().any(axis=1)
nps     = nonpen[has_pl].reset_index(drop=True)
X_ps    = nps[PSXG_FEATURES].fillna(0).astype(float)
y_ps    = nps['is_goal'].astype(int); g_ps = nps['match_id']
print(f'  Shots with placement: {len(nps):,}/{len(nonpen):,}')
g1 = GroupShuffleSplit(1,test_size=0.20,random_state=42)
dp,tp = next(g1.split(X_ps,y_ps,groups=g_ps))
g2 = GroupShuffleSplit(1,test_size=0.25,random_state=0)
trp,cap = next(g2.split(X_ps.iloc[dp],y_ps.iloc[dp],groups=g_ps.iloc[dp]))
psxg_model,psxg_p,psxg_fpr,psxg_tpr,psxg_auc,psxg_brier,psxg_metrics = train_model(
    X_ps.iloc[dp].iloc[trp],y_ps.iloc[dp].iloc[trp],g_ps.iloc[dp].iloc[trp],
    X_ps.iloc[dp].iloc[cap],y_ps.iloc[dp].iloc[cap],X_ps.iloc[tp],y_ps.iloc[tp],'psxG')
X_all_ps = nonpen[PSXG_FEATURES].fillna(0).astype(float)
nonpen['psxg'] = nonpen['xg'].copy()
nonpen.loc[has_pl,'psxg'] = psxg_model.predict_proba(X_all_ps[has_pl])[:,1]
shots.loc[~pen_mask,'psxg'] = nonpen['psxg'].values
joblib.dump(psxg_model, os.path.join(OUTPUT_DIR,'model_psxg.pkl'))

print('\n── Model 3: Situation Danger ──')
sit_model,sit_p,sit_fpr,sit_tpr,sit_auc,sit_brier,sit_metrics = train_model(
    X_sit_tr,y_tr,g_tr,X_sit_cal,y_cal,X_sit_test,y_test,'Situation')
nonpen['situation_danger'] = sit_model.predict_proba(X_sit)[:,1]
shots.loc[~pen_mask,'situation_danger'] = nonpen['situation_danger'].values
joblib.dump(sit_model, os.path.join(OUTPUT_DIR,'model_situation.pkl'))

print('\n── Model 4: P(On Target) ──')
pot_model,pot_p,pot_fpr,pot_tpr,pot_auc,pot_brier,pot_metrics = train_model(
    X_pot_tr,y_ot_tr,g_tr,X_pot_cal,y_ot_cal,X_pot_test,y_ot_test,'P(OT)')
nonpen['p_ontarget'] = pot_model.predict_proba(X_pot)[:,1]
shots.loc[~pen_mask,'p_ontarget'] = nonpen['p_ontarget'].values
joblib.dump(pot_model, os.path.join(OUTPUT_DIR,'model_pontarget.pkl'))

print('\n── Model 5: xGOT ──')
ot_mask = nonpen['is_on_target']==1
not_    = nonpen[ot_mask].reset_index(drop=True)
X_ot    = not_[PSXG_FEATURES].fillna(0).astype(float)
y_otg   = not_['is_goal'].astype(int); g_ot = not_['match_id']
print(f'  On-target: {len(not_):,}  Goals: {y_otg.sum():,} ({y_otg.mean()*100:.1f}%)')
go = GroupShuffleSplit(1,test_size=0.20,random_state=42)
do,to = next(go.split(X_ot,y_otg,groups=g_ot))
gi = GroupShuffleSplit(1,test_size=0.25,random_state=0)
tri,cai = next(gi.split(X_ot.iloc[do],y_otg.iloc[do],groups=g_ot.iloc[do]))
xgot_model,xgot_p,xgot_fpr,xgot_tpr,xgot_auc,xgot_brier,xgot_metrics = train_model(
    X_ot.iloc[do].iloc[tri],y_otg.iloc[do].iloc[tri],g_ot.iloc[do].iloc[tri],
    X_ot.iloc[do].iloc[cai],y_otg.iloc[do].iloc[cai],X_ot.iloc[to],y_otg.iloc[to],'xGOT')
nonpen['xgot'] = 0.0
nonpen.loc[ot_mask,'xgot'] = xgot_model.predict_proba(X_ot)[:,1]
shots.loc[~pen_mask,'xgot'] = nonpen['xgot'].values
shots.loc[pen_mask,'xgot']  = pen_xg
joblib.dump(xgot_model, os.path.join(OUTPUT_DIR,'model_xgot.pkl'))

print('\n✓ All five models trained and saved')

# === Original notebook cell 16 ===
shots['placement_quality'] = shots['psxg'] - shots['xg']
shots['execution_quality'] = shots['xg']  - shots['situation_danger']
shots['ot_over_exp']       = shots['is_on_target'] - shots.get('p_ontarget', 0)
shots['save_difficulty']   = np.where(shots['is_blocked']==1, shots['psxg'], np.nan)
shots['finishing_luck']    = shots['is_goal'] - shots['psxg']

def shot_power(row):
    p = 50.0
    if row['is_volley']:      p += 25
    if row['is_half_volley']:  p += 12
    if row['is_header']:      p -= 8
    if row['is_other_body_part']:      p -= 12
    # One-on-one is context, not a direct shot-power adjustment
    if row['is_deflected']:   p += 5
    return float(np.clip(p, 0, 100))

def contact_quality(psxg, tech, ps):
    if pd.isna(psxg) or pd.isna(ps): return float(psxg*100) if not pd.isna(psxg) else np.nan
    return float(np.clip((0.50*(ps/100) + 0.30*psxg + 0.20*(tech/60))*100, 0, 100))

shots['shot_power']      = shots.apply(shot_power, axis=1)
shots['contact_quality'] = shots.apply(
    lambda r: contact_quality(r['psxg'], r['technique_index'], r['placement_score']), axis=1)

for col in ['xg','psxg','situation_danger','p_ontarget','xgot']:
    if col in shots.columns:
        mn, mx = shots[col].min(), shots[col].max()
        shots[f'{col}_index'] = (shots[col]-mn)/max(mx-mn,1e-9)*100

print('Derived metrics added ✓')
print(shots[['player_name','season','xg','psxg','xgot','placement_score','shot_power','contact_quality','goal_zone']].tail(5).to_string(index=False))

# === Original notebook cell 18 ===
MO_FEATURES = XG_FEATURES.copy()
OUTCOME_MAP  = {13:0, 14:1, 15:2, 16:3}
OUTCOME_NAME = {0:'Miss', 1:'Post', 2:'Save', 3:'Goal'}

X_mo = nonpen[MO_FEATURES].fillna(0).astype(float)
y_mo = nonpen['type_id'].map(OUTCOME_MAP).astype(int)
g_mo = nonpen['match_id']

gm = GroupShuffleSplit(1,test_size=0.20,random_state=42)
dm,tm = next(gm.split(X_mo,y_mo,groups=g_mo))
gm2 = GroupShuffleSplit(1,test_size=0.25,random_state=0)
trm,cam = next(gm2.split(X_mo.iloc[dm],y_mo.iloc[dm],groups=g_mo.iloc[dm]))

mo_clf = xgb.XGBClassifier(
    objective='multi:softprob', num_class=4, eval_metric='mlogloss',
    use_label_encoder=False, max_depth=5, learning_rate=0.05,
    n_estimators=400, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=5, random_state=42, verbosity=0, n_jobs=-1,
)
mo_clf.fit(X_mo.iloc[dm].iloc[trm], y_mo.iloc[dm].iloc[trm])
test_proba = mo_clf.predict_proba(X_mo.iloc[tm])
print(f'Log-loss: {log_loss(y_mo.iloc[tm], test_proba):.4f}')
for c, name in OUTCOME_NAME.items():
    yb = (y_mo.iloc[tm]==c).astype(int)
    if yb.sum() > 0:
        print(f'  {name}: AUC={roc_auc_score(yb, test_proba[:,c]):.4f}')

all_proba = mo_clf.predict_proba(X_mo)
for i, col in enumerate(['p_miss','p_post','p_save','p_goal_mo']):
    shots.loc[~pen_mask, col] = all_proba[:, i]
joblib.dump(mo_clf, os.path.join(OUTPUT_DIR,'model_multi_outcome.pkl'))
print('Multi-outcome model saved ✓')

# === Original notebook cell 20 ===
# ── Shot archetypes (k-means, k=6) ──────────────────────────────────────────
CLUSTER_FEATURES = ['x','y_sym','log_distance','angle_sin',
                    'is_header','is_half_volley','is_volley','is_one_on_one',
                    'goal_y_norm','goal_h_norm']
cd = shots[CLUSTER_FEATURES].copy()
cd['goal_y_norm'] = cd['goal_y_norm'].clip(-1,1).fillna(0.0)
cd['goal_h_norm'] = cd['goal_h_norm'].clip(0,1).fillna(0.35)

scaler = StandardScaler()
X_cl   = scaler.fit_transform(cd.fillna(0).values)
km     = KMeans(n_clusters=6, random_state=42, n_init=20)
shots['archetype_id'] = km.fit_predict(X_cl)

centres = scaler.inverse_transform(km.cluster_centers_)
cdf     = pd.DataFrame(centres, columns=CLUSTER_FEATURES)
def name_arch(row):
    if row['is_header'] > 0.4:          return 'Aerial Header'
    if row['x'] >= 88 and row['y_sym'] < 6: return 'Close-Range Central'
    if row['log_distance'] > np.log(22): return 'Long-Range Effort'
    if row['is_one_on_one'] > 0.5:     return 'One-on-One'
    if row['is_half_volley'] > 0.25 or row['is_volley'] > 0.25: return 'First-Time / Volley'
    return 'Placed Box Finish'
archetype_names = {i: name_arch(cdf.iloc[i]) for i in range(6)}
seen = {}
for k, v in archetype_names.items():
    n = seen.get(v, 0)
    archetype_names[k] = f'{v} ({n+1})' if n > 0 else v
    seen[v] = n + 1
shots['archetype'] = shots['archetype_id'].map(archetype_names)

print('Archetypes:')
print(shots.groupby('archetype')[['is_goal','xg','psxg','shot_power']]
        .agg({'is_goal':['count','mean'],'xg':'mean','psxg':'mean','shot_power':'mean'})
        .round(3))

# ── Rolling form ─────────────────────────────────────────────────────────────
shots_s = shots.sort_values(['player_name','season','time_min']).copy()
for col, src in [('rolling_xg10','xg'),('rolling_psxg10','psxg'),('rolling_goals10','is_goal')]:
    shots_s[col] = shots_s.groupby('player_name')[src].transform(
        lambda s: s.rolling(10, min_periods=3).mean())
shots = shots_s.sort_index()
print('Rolling form computed ✓')

# === Original notebook cell 23 ===
# ── 10b · Empirical Bayes Beta-Binomial  (fast) ─────────────────────────────
#
# Model:   goals_i | n_i, p_i  ~  Binomial(n_i, p_i)
#          p_i                  ~  Beta(α, β)          ← league prior
#
# Posterior for player i: Beta(α + goals_i, β + n_i - goals_i)
# → posterior mean = (α + goals_i) / (α + β + n_i)   (shrinkage estimate)
#
# α, β estimated from data by maximising the marginal Beta-Binomial log-likelihood.
# ─────────────────────────────────────────────────────────────────────────────

# Aggregate per player (career, non-penalty)
eb_df = (
    shots[shots['is_penalty'] == 0]
    .groupby('player_name')
    .agg(n=('is_goal', 'count'), goals=('is_goal', 'sum'), xg_sum=('xg', 'sum'))
    .reset_index()
)
eb_df = eb_df[eb_df['n'] >= 5].copy()
eb_df['raw_rate'] = eb_df['goals'] / eb_df['n']
eb_df['xg_rate']  = eb_df['xg_sum'] / eb_df['n']   # avg xG per shot

# ── Fit Beta prior via marginal log-likelihood ────────────────────────────────
def bb_nll(params, n_arr, k_arr):
    """Negative Beta-Binomial log-likelihood."""
    a, b = np.exp(params)   # keep positive
    ll = 0.0
    for ni, ki in zip(n_arr, k_arr):
        ll += (betaln(ki + a, ni - ki + b) - betaln(a, b))
    return -ll

res = minimize(
    bb_nll,
    x0=[0.0, 2.5],
    args=(eb_df['n'].values, eb_df['goals'].values),
    method='Nelder-Mead',
    options={'maxiter': 5000, 'xatol': 1e-6}
)
alpha_hat, beta_hat = np.exp(res.x)
league_mean = alpha_hat / (alpha_hat + beta_hat)
print(f'Beta prior: α={alpha_hat:.3f}  β={beta_hat:.3f}  '
      f'(league mean = {league_mean:.4f}  ≈ {league_mean*100:.2f}%)')

# ── Posterior estimates ────────────────────────────────────────────────────────
eb_df['eb_alpha_post'] = alpha_hat + eb_df['goals']
eb_df['eb_beta_post']  = beta_hat  + (eb_df['n'] - eb_df['goals'])
eb_df['eb_mean']       = eb_df['eb_alpha_post'] / (eb_df['eb_alpha_post'] + eb_df['eb_beta_post'])
eb_df['eb_ci_lo']      = stats.beta.ppf(0.05, eb_df['eb_alpha_post'], eb_df['eb_beta_post'])
eb_df['eb_ci_hi']      = stats.beta.ppf(0.95, eb_df['eb_alpha_post'], eb_df['eb_beta_post'])
eb_df['eb_vs_xg']      = eb_df['eb_mean'] - eb_df['xg_rate']   # Bayesian finishing edge
eb_df['shrinkage']     = 1 - (eb_df['eb_mean'] - league_mean) / (
                             (eb_df['raw_rate'] - league_mean).replace(0, np.nan))

eb_df = eb_df.sort_values('eb_mean', ascending=False)
print(f'\nTop 15 finishers by Empirical-Bayes conversion rate (min 5 shots):')
print(eb_df[['player_name','n','goals','raw_rate','xg_rate','eb_mean','eb_ci_lo','eb_ci_hi','eb_vs_xg']].head(15).round(4).to_string(index=False))

# === Optional original PyMC hierarchical model (disabled by default) ===
if RUN_PYMC_HIERARCHICAL:
    for pkg, imp in [('pymc', 'pymc'), ('arviz', 'arviz')]:
        try:
            __import__(imp)
        except ImportError:
            _pip(pkg)
    import pymc as pm
    import arviz as az
    # ── 10c · PyMC Hierarchical Logistic Regression  (optional ~3-8 min) ─────────
    #
    # Model (non-centred parameterisation):
    #
    #   σ_δ            ~ HalfNormal(0.5)           ← league spread in finishing skill
    #   δ_offset_i     ~ Normal(0, 1)  for i in players
    #   δ_i            = σ_δ · δ_offset_i           ← per-player finishing skill
    #
    #   logit(p_ij)    = logit(xG_ij) + δ_i         ← xG as a fixed offset
    #   goal_ij        ~ Bernoulli(p_ij)
    #
    # Trained at PLAYER level (aggregated over shots) for speed.
    # Shot-level model would be identical but ~100x slower with this dataset.
    # ─────────────────────────────────────────────────────────────────────────────

    MIN_SHOTS_PYMC = 10   # minimum shots to include a player

    pymc_df = (
        shots[shots['is_penalty'] == 0]
        .groupby('player_name')
        .agg(n=('is_goal','count'), goals=('is_goal','sum'), xg_mean=('xg','mean'))
        .reset_index()
    )
    pymc_df = pymc_df[pymc_df['n'] >= MIN_SHOTS_PYMC].reset_index(drop=True)
    n_players = len(pymc_df)

    # Convert mean xG to logit offset, clipped for numerical stability
    xg_logit_offset = np.log(
        np.clip(pymc_df['xg_mean'].values, 0.001, 0.999) /
        (1 - np.clip(pymc_df['xg_mean'].values, 0.001, 0.999))
    )

    print(f'Fitting PyMC model: {n_players} players, min {MIN_SHOTS_PYMC} shots each')
    print('This takes ~3-8 minutes on Colab CPU...\n')

    with pm.Model() as hier_model:
        # ── Hyperprior: league-wide spread of finishing skill ──────────────────
        sigma_delta = pm.HalfNormal('sigma_delta', sigma=0.5)

        # ── Per-player skill offsets (non-centred) ────────────────────────────
        delta_offset = pm.Normal('delta_offset', mu=0, sigma=1, shape=n_players)
        delta        = pm.Deterministic('delta', sigma_delta * delta_offset)

        # ── Shot probability: xG logit + player skill ─────────────────────────
        logit_p = xg_logit_offset + delta
        p       = pm.Deterministic('p', pm.math.sigmoid(logit_p))

        # ── Likelihood ────────────────────────────────────────────────────────
        goals_obs = pm.Binomial(
            'goals_obs',
            n  = pymc_df['n'].values,
            p  = p,
            observed = pymc_df['goals'].values,
        )

        # ── Sample ────────────────────────────────────────────────────────────
        trace = pm.sample(
            draws=1000, tune=500,
            target_accept=0.90,
            return_inferencedata=True,
            progressbar=True,
            random_seed=42,
        )

    print('\n✓ Sampling complete')
    print(az.summary(trace, var_names=['sigma_delta'], round_to=4).to_string())

# === Original notebook cell 27 ===
# ── Player summary ──────────────────────────────────────────────────────────
# Fill row-level suppression values after model predictions are available.
shots['shot_suppression_value'] = 1
shots['xg_suppression_value'] = shots['xg']
shots['psxg_suppression_value'] = shots['psxg']

player_summary = (
    shots.groupby(['player_name','season'])
    .agg(
        shots_n     = ('is_goal','count'),
        goals       = ('is_goal','sum'),
        xg          = ('xg','sum'),
        psxg        = ('psxg','sum'),
        xgot        = ('xgot','sum'),
        placement_mean = ('placement_score','mean'),
        power_mean  = ('shot_power','mean'),
        contact_mean = ('contact_quality','mean'),
    )
    .reset_index()
)
player_summary['conv_rate']       = player_summary['goals'] / player_summary['shots_n']
player_summary['goals_minus_xg']  = player_summary['goals'] - player_summary['xg']
player_summary['goals_minus_psxg']= player_summary['goals'] - player_summary['psxg']

career = (
    shots.groupby('player_name')
    .agg(
        shots_n     = ('is_goal','count'),
        goals       = ('is_goal','sum'),
        xg          = ('xg','sum'),
        psxg        = ('psxg','sum'),
        xgot        = ('xgot','sum'),
        seasons     = ('season','nunique'),
    )
    .reset_index()
)
career['conv_rate']        = career['goals'] / career['shots_n']
career['goals_minus_xg']   = career['goals'] - career['xg']
career['goals_minus_psxg'] = career['goals'] - career['psxg']

# Team shot suppression: one row per team-match with shots/xG for and against.
team_for = (
    shots.groupby(['season', 'match_id', 'match_file', 'contestant_id'])
    .agg(
        shots_for=('is_goal', 'count'),
        goals_for=('is_goal', 'sum'),
        xg_for=('xg', 'sum'),
        psxg_for=('psxg', 'sum'),
        xgot_for=('xgot', 'sum'),
    )
    .reset_index()
)
team_against = (
    shots.groupby(['season', 'match_id', 'match_file', 'opponent_contestant_id'])
    .agg(
        shots_against=('is_goal', 'count'),
        goals_against=('is_goal', 'sum'),
        xg_against=('xg', 'sum'),
        psxg_against=('psxg', 'sum'),
        xgot_against=('xgot', 'sum'),
    )
    .reset_index()
    .rename(columns={'opponent_contestant_id': 'contestant_id'})
)
team_shot_suppression = team_for.merge(
    team_against,
    on=['season', 'match_id', 'match_file', 'contestant_id'],
    how='outer',
).fillna(0)
team_shot_suppression['shot_suppression_net'] = (
    team_shot_suppression['shots_against'] - team_shot_suppression['shots_for']
)
team_shot_suppression['xg_suppression_net'] = (
    team_shot_suppression['xg_against'] - team_shot_suppression['xg_for']
)

shot_assist_summary = (
    shots[shots['has_shot_assist'].eq(1) & shots['shot_assist_player_id'].ne('')]
    .groupby(['season', 'shot_assist_player_id', 'shot_assist_player_name'])
    .agg(
        assisted_shots=('is_goal', 'count'),
        assisted_goals=('is_goal', 'sum'),
        assisted_xg=('xg', 'sum'),
        assisted_psxg=('psxg', 'sum'),
    )
    .reset_index()
    .sort_values(['assisted_xg', 'assisted_shots'], ascending=False)
)

# ── One-on-one efficiency ───────────────────────────────────────────────────────
peff = shots.groupby(['season','is_one_on_one']).agg(
    shots_n=('is_goal','count'), goals=('is_goal','sum'),
    xg_sum=('xg','sum'), psxg_sum=('psxg','sum'),
).reset_index()
peff['conv_rate'] = peff['goals'] / peff['shots_n']
peff['xg_per_shot'] = peff['xg_sum'] / peff['shots_n']
peff_p = peff.copy()

shots.to_csv(os.path.join(OUTPUT_DIR,'all_shots_full.csv'), index=False)
player_summary.to_csv(os.path.join(OUTPUT_DIR,'player_summary.csv'), index=False)
career.to_csv(os.path.join(OUTPUT_DIR,'career_summary.csv'), index=False)
team_shot_suppression.to_csv(os.path.join(OUTPUT_DIR,'team_shot_suppression.csv'), index=False)
shot_assist_summary.to_csv(os.path.join(OUTPUT_DIR,'shot_assist_summary.csv'), index=False)
print(f'Data saved: {len(shots):,} shots, {len(career):,} players')
print(f'Seasons: {sorted(shots["season"].unique())}')

# === Original notebook cell 48 ===
# Final save
shots.to_csv(os.path.join(OUTPUT_DIR,'all_shots_full.csv'), index=False)
player_summary.to_csv(os.path.join(OUTPUT_DIR,'player_summary.csv'), index=False)
career.to_csv(os.path.join(OUTPUT_DIR,'career_summary.csv'), index=False)
team_shot_suppression.to_csv(os.path.join(OUTPUT_DIR,'team_shot_suppression.csv'), index=False)
shot_assist_summary.to_csv(os.path.join(OUTPUT_DIR,'shot_assist_summary.csv'), index=False)
peff_p.to_csv(os.path.join(OUTPUT_DIR,'one_on_one_efficiency.csv'), index=False)
eb_df.to_csv(os.path.join(OUTPUT_DIR,'bayesian_finishing.csv'), index=False)

print('Files saved to', OUTPUT_DIR)
print()
files_out = glob.glob(os.path.join(OUTPUT_DIR,'*'))
for f in sorted(files_out):
    size = os.path.getsize(f)/1024
    print(f'  {os.path.basename(f):40s}  {size:8.1f} KB')

print(f'\n✓ Done. {len(shots.columns)} columns in all_shots_full.csv')


Dependencies ready ✓
All non-visual packages loaded ✓
Output dir: /Users/marclamberts/Downloads/Danger Model/xg_output
xG variant: objective (Big Chance excluded)
Helpers defined ✓
load_match defined with game state, shot-assist links and suppression context ✓
Found 2524 JSON files across 9 season roots
Loaded 64,609 shots | Goals: 6,893 (10.7%)
Unique matches: 2,524
Seasons: ['Eredivisie', 'Unknown']
                                                                                             match_id                                                     match_file match_date                                                match_name   home_team_name              away_team_name   event_id  event_index  season                 player_id player_name             contestant_id    opponent_contestant_id  type_id  period_id  time_min  time_sec  time_seconds                timestamp  team_goals_before  opponent_goals_before  goal_diff_before game_state  team_goals_after  opponent_goals_after  goa